# Event Management Chatbot

This notebook provides a conversational interface for managing events using natural language.

In [1]:
# Setup and imports
import sys
sys.path.append('..')  # Add src to path

import ipywidgets as widgets
from IPython.display import display, HTML
from src.event_system import EventSystem
from src.ui.interface import ChatInterface

# Initialize system
system = EventSystem()
interface = ChatInterface(system)

# Display the interface
interface.display()

In [2]:
# Direct test of selection provider
# Get the provider from your system (it's already imported internally)
provider = system.orchestrator.clarifier.selection_provider

print("Provider type:", type(provider).__name__)

# Test context similar to what would be passed
test_context = {
    "task_details": {
        "action": "schedule_session",
        "parameters": {
            "expected_attendees": 30,
            "requires_av": False,
            "hosted_by": "Krishna"
        }
    }
}

# Call get_options directly
print("Testing selection provider directly:")
options = provider.get_options("venue_selection", test_context)
print(f"Got {len(options)} options")
print("First 5:", options[:5] if len(options) > 5 else options)

# Also check what venues match the criteria
state = system.get_current_state()
suitable = []
for name, props in state.get('venues', {}).items():
    if props.get('capacity', 0) >= 30:
        suitable.append(name)
print(f"\nManual check: {len(suitable)} venues have capacity >= 30")
print("First 5 suitable:", suitable[:5])

Provider type: StateBasedSelectionProvider
Testing selection provider directly:
[DEBUG] get_options called with selection_type: venue_selection
[DEBUG] Context: {'task_details': {'action': 'schedule_session', 'parameters': {'expected_attendees': 30, 'requires_av': False, 'hosted_by': 'Krishna'}}}
[DEBUG] State has 60 venues
[DEBUG] Task details: {'action': 'schedule_session', 'parameters': {'expected_attendees': 30, 'requires_av': False, 'hosted_by': 'Krishna'}}
[DEBUG] Looking for venues with capacity >= 30, AV required: False
[DEBUG] Found 22 suitable venues out of 60 total
[DEBUG] _get_suitable_venues returned: ['Automatically Assigned Venue Name', 'Room 2', 'Room 5']
Got 22 options
First 5: ['Automatically Assigned Venue Name', 'Room 2', 'Room 5', 'The Innovation Hub', 'Workshop C']

Manual check: 42 venues have capacity >= 30
First 5 suitable: ['Main Auditorium', 'Workshop A', 'Workshop B', 'Lecture Hall', 'Grand Ballroom']


In [8]:
# Debug step 1: Check what's in the orchestrator's schema
print("=== ORCHESTRATOR SCHEMA CHECK ===")
schedule_schema = system.orchestrator.schema.get("schedule_session", {})
in_venue_type = schedule_schema.get("param_types", {}).get("in_venue", {})
print(f"Orchestrator schema for in_venue: {in_venue_type}")

# Debug step 2: Trace through the actual clarification generation
print("\n=== TRACE CLARIFICATION GENERATION ===")

# Simulate what happens in _request_clarification
task_details = {
    "action": "schedule_session",
    "parameters": {
        "name": "Test Session",
        "hosted_by": "Krishna",
        "expected_attendees": 30
    }
}

missing_params = ["in_venue", "requires_av"]

# Call the clarifier directly
clarifier = system.orchestrator.clarifier
print(f"Clarifier type: {type(clarifier).__name__}")

# Check if generate_message gets the schema
import inspect
sig = inspect.signature(clarifier.generate_message)
print(f"generate_message parameters: {list(sig.parameters.keys())}")

# Call generate_message directly
clarification_data = clarifier.generate_message(
    missing_params,
    task_details,
    "admin",
    "llama3:8b",
    system.orchestrator.connector,
    system.orchestrator.schema  # Is this being passed?
)

print(f"\nGenerated clarification data:")
for field in clarification_data.get("form_fields", []):
    print(f"  Field: {field['name']}, Type: {field['type']}")
    if field.get('options'):
        print(f"    Has {len(field['options'])} options")

=== ORCHESTRATOR SCHEMA CHECK ===
Orchestrator schema for in_venue: {'type': 'venue_selection', 'dsl_keyword': 'in_venue'}

=== TRACE CLARIFICATION GENERATION ===
Clarifier type: ClarificationGenerator
generate_message parameters: ['missing_params', 'task_details', 'role', 'model_name', 'connector', 'schema']
[DEBUG] get_options called with selection_type: venue_selection
[DEBUG] Context: {'task_details': {'action': 'schedule_session', 'parameters': {'name': 'Test Session', 'hosted_by': 'Krishna', 'expected_attendees': 30}}, 'action': 'schedule_session', 'param': 'in_venue'}
[DEBUG] State has 60 venues
[DEBUG] Task details: {'action': 'schedule_session', 'parameters': {'name': 'Test Session', 'hosted_by': 'Krishna', 'expected_attendees': 30}}
[DEBUG] Looking for venues with capacity >= 30, AV required: False
[DEBUG] Found 22 suitable venues out of 60 total
[DEBUG] _get_suitable_venues returned: ['Automatically Assigned Venue Name', 'Room 2', 'Room 5']

Generated clarification data:
  F